In [ ]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

import regex as re
print(re.findall(PAT, "some text that i'll pre-tokenize"))
print(re.finditer)

In [ ]:
from typing import BinaryIO
import os

def find_chunk_boundaries(
    file: BinaryIO,
    chunk_num: int,
    special_token: bytes
) -> list[int]:
    assert isinstance(special_token, bytes), "token must bytes"

    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // chunk_num
    boundaries = [i * chunk_size for i in range(chunk_num + 1)]
    boundaries[-1] = file_size

    mini_chunk_size = 4096
    for idx in range(1, len(boundaries) - 1):
        init_position = boundaries[idx]
        file.seek(init_position)
        
        while True:
            mini_chunk = file.read(mini_chunk_size)

            if mini_chunk == b"":
                boundaries[idx] = file_size
                break

            special_at = mini_chunk.find(special_token)
            if special_at != -1:
                boundaries[idx] = init_position + special_at
                break
            init_position += mini_chunk_size
    return sorted(set(boundaries))

with open("../data/tinystories_mini_train.txt", "rb") as f:
    num_process = 4
    chunk_bodundaries = find_chunk_boundaries(f, num_process, b"<|endoftext|>")
    # print(chunk_bodundaries)

    for start, end in zip(chunk_bodundaries[:-1], chunk_bodundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8")
        print(chunk)

In [ ]:
# BPE 朴素实现
def train_bpe(
    input_path: str,
    vocab_size: int,
    special_tokens: list[str]
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    vocab = []
    for i in range(256):
        # 直接bytes(int)是生成 整数位长度的字节串！！
        vocab.append(bytes([i]))
    for i in range(len(special_tokens)):
        vocab.append(special_tokens[i].encode("utf-8"))
    
    merges = []

    with open(input_path, "r") as f:
        text = f.read()
    
    # 处理输入数据
    special_tokens_escape = [re.escape(t) for t in special_tokens]
    pattern = "|".join(special_tokens_escape)
    text_list = re.split(pattern, text)

    # 清洗空白字符串
    text_list = [s for s in text_list if s]

    # 构建 ("l", "o", "w") -> num 的映射
    process = defaultdict(int)
    # print(text_list)
    GPT2_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\w+| ?\d+| ?[^\s\w\d]+|\s+(?!\S)|\s+"""
    for text_s in text_list:
        print(re.findall(GPT2_PATTERN, text_s))
        for chunk in re.findall(GPT2_PATTERN, text_s):
            text_chunk = tuple([bytes([b]) for b in chunk.encode("utf-8")])

            process[text_chunk] += 1

    # 确定提取多少个new token
    merge_num = vocab_size - 256 - len(special_tokens)
    while merge_num:
        # 统计相邻token数
        token_neibo_num = defaultdict(int)
        for k, v in process.items():
            for idx in range(len(k) - 1):
                pair = (k[idx], k[idx + 1])
                token_neibo_num[pair] += v

        # print(token_neibo_num.values())
        # 提取最大且字典序大bytes合并
        best_pair = max(token_neibo_num, key=lambda p: (token_neibo_num[p], p))

        merges.append(best_pair)
        vocab.append(best_pair[0] + best_pair[1])
        
        # 更新process 为下一次寻找做准备
        process_next = defaultdict(int)
        update_token = merges[-1]
        for k, v in process.items():
            new_key = []
            idx = 0

            while idx < len(k) - 1:
                if update_token[0] == k[idx] and update_token[1] == k[idx + 1]:
                    new_key.append(k[idx] + k[idx + 1])
                    idx += 2
                else:
                    new_key.append(k[idx])
                    idx += 1
            
            if idx < len(k):
                new_key.append(k[-1])
            process_next[tuple(new_key)] += v

        process = process_next.copy()    

        merge_num -= 1
    vocab_dict = {idx: token for idx, token in enumerate(vocab)}
    return vocab_dict, merges

In [ ]:
# BPE 索引加速
# 每次更新pair都会对所有的token进行遍历，
# 但每次合并后更新只会影响周围重叠的pair，
# 只要索引一下被影响的token，直接处理即可

# token_neibo_num保存pair数量
token_neibo_num = {
    (b'lo'): 3,
    (b'ow'): 2,
    ...
}
# token_affected_seq用于索引pari影响的token
token_affected_seq = {
    (b'lo'): set((b'l', b'o', b'w'), (b'l', b'o', b'w'....)),
    (b'ow'): set((b'l', b'o', b'w'), (b'l', b'o', b'w'....)),
    ...
}
# 实现思路为：
# 1. 找出合并pair
# 2. 删除旧token贡献
# 3. 合并后token为什么形式
# 4. 添加新token贡献
def train_bpe(
    input_path: str,
    vocab_size: int,
    special_tokens: list[str]
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    vocab = []
    for i in range(256):
        # 直接bytes(int)是生成 整数位长度的字节串！！
        vocab.append(bytes([i]))
    for i in range(len(special_tokens)):
        vocab.append(special_tokens[i].encode("utf-8"))
    
    merges = []

    with open(input_path, "r") as f:
        text = f.read()
    
    # 处理输入数据
    special_tokens_escape = [re.escape(t) for t in special_tokens]
    pattern = "|".join(special_tokens_escape)
    text_list = re.split(pattern, text)

    # 清洗空白字符串
    text_list = [s for s in text_list if s]

    # 构建 ("l", "o", "w") -> num 的映射
    process = defaultdict(int)
    # print(text_list)
    GPT2_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\w+| ?\d+| ?[^\s\w\d]+|\s+(?!\S)|\s+"""
    for text_s in text_list:
        print(re.findall(GPT2_PATTERN, text_s))
        for chunk in re.findall(GPT2_PATTERN, text_s):
            text_chunk = tuple([bytes([b]) for b in chunk.encode("utf-8")])
            process[text_chunk] += 1

    token_neibo_num = defaultdict(int)
    token_affected_seq = defaultdict(set)

    for k, v in process.items():
        for idx in range(len(k) - 1):
            pair = (k[idx], k[idx + 1])
            token_neibo_num[pair] += v
            token_affected_seq[pair].add(k)

    # 确定提取多少个new token
    merge_num = vocab_size - 256 - len(special_tokens)
    while merge_num:
        # 提取最大且字典序大bytes合并
        best_pair = max(token_neibo_num, key=lambda p: (token_neibo_num[p], p))

        merges.append(best_pair)
        vocab.append(best_pair[0] + best_pair[1])
        # print(best_pair)
        affected_seq = token_affected_seq[best_pair]
        # print(affected_seq)
        
        for old_key in affected_seq.copy():
            v = process[old_key]
            # 删除旧key贡献
            for idx in range(len(old_key) - 1):
                pair = (old_key[idx], old_key[idx + 1])
                token_neibo_num[pair] -= v
                token_affected_seq[pair].discard(old_key)

            # 计算合并后
            new_key = []
            idx = 0
            while idx < len(old_key) - 1:
                if old_key[idx] == best_pair[0] and old_key[idx + 1] == best_pair[1]:
                    new_key.append(best_pair[0] + best_pair[1])
                    idx += 2
                else:
                    new_key.append(old_key[idx])
                    idx += 1
            if idx != len(old_key):
                new_key.append(old_key[-1])
            
            # 计算新key 贡献
            new_key_tuple = tuple(new_key)
            # print(new_key_tuple)
            for idx in range(len(new_key) - 1):
                pair = (new_key_tuple[idx], new_key_tuple[idx + 1])
                token_neibo_num[pair] += v
                token_affected_seq[pair].add(new_key_tuple)
           
           # 更新process
            process[old_key] -= v
            process[new_key_tuple] = process.get(new_key_tuple, 0) + v
            # print(process)
        
        del token_neibo_num[best_pair]
        del token_affected_seq[best_pair]
        process = {k: v for k, v in process.items() if v > 0}
        merge_num -= 1
    vocab_dict = {idx: token for idx, token in enumerate(vocab)}
    return vocab_dict, merges

In [12]:
GPT2_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\w+| ?\d+| ?[^\s\w\d]+|\s+(?!\S)|\s+"""
import regex as re

s = "hi i am bob!"
print(re.findall(GPT2_PATTERN, s))



['hi', ' i', ' am', ' bob', '!']


In [ ]:
a = "aab"
b = "aac"
l = [a, b]
dic = {
    "aab": [5, 33],
    "aac": 5,
    "aba": 3
}
print(dic["aab"][1] + 1)

c = b"124"
print(c[1:])
